In [1]:
import numpy as np
import torch
import hockey.hockey_env as h_env

from memory import ReplayBuffer
from sac import SACAgent

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
env = h_env.HockeyEnv()
player2 = h_env.BasicOpponent(weak=False)

ac_space = env.action_space
o_space = env.observation_space
print(ac_space)
print(o_space)
print(list(zip(env.observation_space.low, env.observation_space.high)))

Box(-1.0, 1.0, (8,), float32)
Box(-inf, inf, (18,), float32)
[(-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf), (-inf, inf)]


In [ ]:
max_episodes=600
max_steps=500 
add_reward_0_to_buffer_proba = 0.1

buffer = ReplayBuffer()
agent = SACAgent(env.observation_space.shape[0], env.action_space.shape[0], noise_seq_len=int(1e5), device = device)

In [4]:
ob,_info = env.reset()
print(ob)
agent.actor(torch.FloatTensor(ob).unsqueeze(0).to(device))

[-3.          0.          0.          0.          0.          0.
  3.          0.          0.          0.          0.          0.
  1.91297674  0.7364974   0.          0.          0.          0.        ]


(tensor([[-0.3049, -0.0056,  0.3271,  0.2417,  0.3662, -0.4165,  0.1979,  0.1278]],
        grad_fn=<AddmmBackward0>),
 tensor([[-0.6555, -0.4363,  0.4342, -0.3043, -0.3238,  0.1238,  0.0405, -0.1997]],
        grad_fn=<ClampBackward1>))

In [5]:
stats = []
losses = []

In [ ]:
for i in range(max_episodes):
    # print("Starting a new episode")    
    total_reward = []
    ob, _info = env.reset()
    obs_agent2 = env.obs_agent_two()
    done = False
    for t in range(max_steps):
        with torch.no_grad():
            a1, _ = agent.actor.sample(torch.FloatTensor(ob).unsqueeze(0).to(device))
        a1 = a1.cpu().numpy()[0]
        a2 = player2.act(obs_agent2)

        (ob_new, reward, done, trunc, _info) = env.step(np.hstack([a1,a2]))
        if reward == 0 and np.random.random() < add_reward_0_to_buffer_proba:
            buffer.add((ob, a1, reward, ob_new, float(done)))
        total_reward.append(reward)
        ob=ob_new        
        if done: 
            break
    agent.update(buffer)
    stats.append([i,total_reward,t+1])
    if (i+1)%20 == 0:
        agent.actor.gen.reset()
    
    if ((i-1)%20==0):
        print("{}: Reward: {}".format(i, np.sum(total_reward)))

1: Reward: 0.0
21: Reward: 0.0
41: Reward: 0.0
61: Reward: -11.541906317058936
81: Reward: 0.0
101: Reward: 0.0
121: Reward: 0.0
141: Reward: 0.0
161: Reward: 0.0
181: Reward: 0.0
201: Reward: 9.370190843705787


KeyboardInterrupt: 

In [7]:
o, info = env.reset()
_ = env.render()
player2 = h_env.BasicOpponent(weak=False)

In [8]:
obs_buffer = []
reward_buffer=[]
obs, info = env.reset()
obs_agent2 = env.obs_agent_two()
for _ in range(251):
    env.render()
    with torch.no_grad():
        a1, _ = agent.actor.sample(torch.FloatTensor(ob).unsqueeze(0).to(device))
        a1 = a1.cpu().numpy()[0]
    a2 = player2.act(obs_agent2)

    obs, r, d, t, info = env.step(np.hstack([a1,a2]))    
    obs_buffer.append(obs)
    reward_buffer.append(r)
    obs_agent2 = env.obs_agent_two()
    if d or t: 
        break
obs_buffer = np.asarray(obs_buffer)
reward_buffer = np.asarray(reward_buffer)

In [9]:
env.close()